In [1]:
import csv, json, os, shutil, hashlib
from collections import Counter

# ---- STUDENT-SPECIFIC VALUE ----
SEED_DIGIT = 6   # <- your SEED_DIGIT

TARGET_YEAR = 2015 + SEED_DIGIT
SRC = r"C:\Users\Nate\Lab6_File_Handling_Linsangan\Lab6_File_handling_Additional_Exercises\archive (1)\Global_Cybersecurity_Threats_2015-2024.csv"
OUT_DIR = "output"
BACKUP_DIR = "backup"
FILTERED = os.path.join(OUT_DIR, "filtered_incidents.csv")
SUMMARY = os.path.join(OUT_DIR, "incident_summary.json")
BACKUP = os.path.join(BACKUP_DIR, "incident_summary_backup.json")
REPORT = os.path.join(OUT_DIR, "incident_report.txt")

LOSS_COL = "Financial Loss (in Million $)"
YEAR_COL = "Year"
TYPE_COL = "Attack Type"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

total_records = 0
invalid = 0
selected = []          # list of (original_row_dict, loss_float), in dataset order
type_counts = Counter()
fieldnames = []

# Req 1: read the CSV and select incidents in the target year
with open(SRC, newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    print("Columns:", fieldnames)  # confirm names match the constants above
    for row in reader:
        total_records += 1
        try:
            year = int((row.get(YEAR_COL) or "").strip())
        except ValueError:
            invalid += 1
            continue
        if year != TARGET_YEAR:
            continue
        # Handle invalid financial values
        try:
            loss = float((row.get(LOSS_COL) or "").strip())
            if loss < 0 or loss != loss:   # negative or NaN
                raise ValueError
        except ValueError:
            invalid += 1
            continue
        selected.append((row, loss))
        # Req 2: count per attack type
        atk = (row.get(TYPE_COL) or "").strip() or "Unknown"
        type_counts[atk] += 1

n = len(selected)

# Req 3: total and average loss (handle empty results)
total_loss = round(sum(l for _, l in selected), 2)
avg_loss = round(total_loss / n, 2) if n else None
if n == 0:
    print(f"No valid incidents found for {TARGET_YEAR}.")

# Req 4: top five by descending loss; sorted() is stable, so ties keep
# the original dataset order
top5_pairs = sorted(selected, key=lambda p: -p[1])[:5]
top5 = [dict(r) for r, _ in top5_pairs]

# Req 5: export filtered CSV
with open(FILTERED, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(r for r, _ in selected)

# Write JSON summary (includes the top-five records)
summary = {
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_records,
    "invalid_records_skipped": invalid,
    "incidents_in_target_year": n,
    "count_per_attack_type": dict(type_counts),
    "total_financial_loss_million_usd": total_loss,
    "average_financial_loss_million_usd": avg_loss,
    "five_costliest_incidents": top5,
    "filtered_csv_file": FILTERED,
}
with open(SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Backup the JSON file and verify it
shutil.copy2(SUMMARY, BACKUP)

def sha256(path):
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()

exists = os.path.exists(BACKUP)
size_match = exists and os.path.getsize(BACKUP) == os.path.getsize(SUMMARY)
content_match = exists and sha256(BACKUP) == sha256(SUMMARY)
backup_ok = exists and size_match and content_match

# Req 6: text report
with open(REPORT, "w", encoding="utf-8") as f:
    f.write(f"CYBERSECURITY INCIDENT REPORT - {TARGET_YEAR}\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Total records in dataset: {total_records}\n")
    f.write(f"Invalid records skipped: {invalid}\n")
    f.write(f"Incidents in {TARGET_YEAR}: {n}\n\n")
    f.write("Incidents per attack type:\n")
    for k, v in type_counts.most_common():
        f.write(f"  {k}: {v}\n")
    f.write(f"\nTotal financial loss: {total_loss} million USD\n")
    avg_txt = f"{avg_loss} million USD" if n else "N/A"
    f.write(f"Average financial loss: {avg_txt}\n\n")
    f.write("Five costliest incidents:\n")
    for i, (r, l) in enumerate(top5_pairs, 1):
        f.write(f"  {i}. {l} M USD | {r.get(TYPE_COL)} | "
                f"{r.get('Country')} | {r.get('Target Industry')}\n")
    f.write("\nOutput files:\n")
    f.write(f"  Filtered CSV: {FILTERED}\n")
    f.write(f"  JSON summary: {SUMMARY}\n")
    f.write(f"  Backup: {BACKUP} ({os.path.getsize(BACKUP)} bytes)\n")
    f.write(f"  Backup verified: {backup_ok}\n")

# Assessment table values
print("Target year:", TARGET_YEAR)
print("Total records in dataset:", total_records)
print("Invalid records skipped:", invalid)
print("Incidents in target year:", n)
print("Count per attack type:", dict(type_counts))
print("Total financial loss (USD millions):", total_loss)
print("Average financial loss (USD millions):", avg_loss)
print("Five costliest incidents:")
for i, (r, l) in enumerate(top5_pairs, 1):
    print(f"  {i}. {l} | {r.get(TYPE_COL)} | {r.get('Country')}")
print("Filtered CSV:", FILTERED)
print("JSON summary:", SUMMARY)
print("--- Backup verification ---")
print("Backup exists:", exists, "| Size match:", size_match,
      "| Content match:", content_match)
print("BACKUP VERIFIED" if backup_ok else "BACKUP FAILED")
print("Report file:", REPORT)

Columns: ['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']
Target year: 2021
Total records in dataset: 3000
Invalid records skipped: 0
Incidents in target year: 299
Count per attack type: {'Phishing': 41, 'SQL Injection': 59, 'Malware': 49, 'DDoS': 47, 'Ransomware': 55, 'Man-in-the-Middle': 48}
Total financial loss (USD millions): 15873.41
Average financial loss (USD millions): 53.09
Five costliest incidents:
  1. 99.81 | DDoS | USA
  2. 99.77 | Ransomware | France
  3. 98.76 | SQL Injection | France
  4. 98.57 | SQL Injection | China
  5. 98.09 | SQL Injection | India
Filtered CSV: output\filtered_incidents.csv
JSON summary: output\incident_summary.json
--- Backup verification ---
Backup exists: True | Size match: True | Content match: True
BACKUP VERIFIED
Report file: output\incident_report.txt
